In [1]:
import os
import csv

In [5]:
dataset_dir = "../dataset" 
output_csv = "dataset.csv" 
classes = { "person": 1, "other": 0 }
rows = []

In [6]:
for class_name, label in classes.items():
    folder = os.path.join(dataset_dir, class_name)
    for filename in os.listdir(folder):
        if filename.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
            filepath = os.path.join(folder, filename)
            rows.append([filepath, label])


In [7]:
with open(output_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["path", "label"])
    writer.writerows(rows)

print("Готово!")


Готово!


In [9]:
import numpy as np
import cv2, os
from tensorflow import keras

In [11]:
# Параметры
IMG_HEIGHT = 96
IMG_WIDTH = 96
DATA_DIR = "../dataset"  # путь к папке с поддиректориями классов

# Загрузка и подготовка данных
classes = ["person", "other"]
X, y = [], []
for label, cls in enumerate(classes):
    folder = os.path.join(DATA_DIR, cls)
    for fname in os.listdir(folder):
        img = cv2.imread(os.path.join(folder, fname), cv2.IMREAD_GRAYSCALE)
        if img is None: 
            continue
        img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))  # масштабируем к 96x96
        X.append(img.astype(np.float32) / 255.0)        # нормализуем пиксели 0..1
        y.append(label)
X = np.array(X).reshape(-1, IMG_HEIGHT, IMG_WIDTH, 1)
y = np.array(y)
print("Loaded", X.shape, "data samples.")


Loaded (162, 96, 96, 1) data samples.


In [12]:

# Разделение на обучение и тест (например 80/20)
indices = np.arange(len(X))
np.random.shuffle(indices)
train_split = int(0.8 * len(X))
X_train, X_test = X[indices[:train_split]], X[indices[train_split:]]
y_train, y_test = y[indices[:train_split]], y[indices[train_split:]]


In [13]:
# Простая CNN модель
model = keras.models.Sequential([
    keras.layers.Conv2D(8, (3,3), activation='relu', input_shape=(IMG_HEIGHT, IMG_WIDTH, 1)),
    keras.layers.MaxPooling2D((2,2)),
    keras.layers.Conv2D(16, (3,3), activation='relu'),
    keras.layers.MaxPooling2D((2,2)),
    keras.layers.Flatten(),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(len(classes), activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

c:\Users\user\Desktop\tinyml\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 94, 94, 8)      │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 47, 47, 8)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 45, 45, 16)     │         1,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 22, 22, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 7744)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │       123,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │            34 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 125,202 (489.07 KB)

 Trainable params: 125,202 (489.07 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:

# Обучение (например, 10 эпох)
model.fit(X_train, y_train, epochs=10, batch_size=8, validation_data=(X_test, y_test))
# Оценка точности на тесте
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print("Test accuracy:", test_acc)


Epoch 1/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step - accuracy: 0.6434 - loss: 0.6718 - val_accuracy: 0.8485 - val_loss: 0.6225
Epoch 2/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9147 - loss: 0.5315 - val_accuracy: 0.9394 - val_loss: 0.4474
Epoch 3/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9147 - loss: 0.3267 - val_accuracy: 0.8788 - val_loss: 0.3095
Epoch 4/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8992 - loss: 0.2251 - val_accuracy: 0.9091 - val_loss: 0.2601
Epoch 5/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9302 - loss: 0.1828 - val_accuracy: 0.9394 - val_loss: 0.1950
Epoch 6/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9225 - loss: 0.1981 - val_accuracy: 0.9394 - val_loss: 0.1999
Epoch 7/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9457 - loss: 0.1633 - val_accuracy: 0.8788 - val_loss: 0.3005
Epoch 8/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9380 - loss: 0.1469 - val_accuracy: 0.9394 - v